In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

### 1.Reading CSV file

In [0]:
df=spark.read.csv("/Volumes/week-7/default/data/Sample - Superstore.csv",
    header=True, inferSchema=True
)
#The given csv file is stored in volumes andd it is read into df

In [0]:
df.show(5)
#displaying top 5 rows ,to ensure it is read correctly

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

### 2.Creation of delta table

In [0]:
from pyspark.sql.functions import col
df=df.select([col(c).alias(c.replace(" ","_")) for c in df.columns])
df.printSchema()
#I am changing the column names because delta table cannot consists of column names with spaces in it

root
 |-- Row_ID: integer (nullable = false)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = false)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Sub-Category: string (nullable = false)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = false)



In [0]:
d_path="/Volumes/week-7/default/data/superstore_delta"
df.write.mode("overwrite").format("delta").save(d_path)
#delta table path is given and writing the data into delta table

In [0]:
dTable=DeltaTable.forPath(spark,d_path)
dTable.toDF().show()
#displaying the data in delta table

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|     5|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|   Consumer|United States|Fort Lauderdale|       Florida|      33311|  South|OFF-ST-10000760|

### 3.Cleaning the data

In [0]:
df=dTable.toDF()

In [0]:
df=df.dropDuplicates()
#dropping the duplicate rows

In [0]:
df=df.fillna(
    {
    "Category":"Unknown",
    "Sub-Category":"Unknown"
})
#filling the null values of category and sub-category columns with Unknown

In [0]:
df=df.fillna(0)
#filling the null values of all other columns with 0

In [0]:
print(df.count())

9994


In [0]:
df.write.format("delta").mode("overwrite").save(d_path)
dTable=DeltaTable.forPath(spark,d_path)
#After cleaning we need to save the data in delta table

### 4.Create Incremental Dataset

In [0]:
inc_data=[
    ("CA-2016-152156","Technology",1200.0,4),
    ("CA-2016-138688","Furniture",850.0,2),
    ("CA-2025-999999","Office Supplies",300.0,5)
]
inc_df=spark.createDataFrame(inc_data,["Order ID","Category","Sales","Quantity"])
inc_df.show()
#Creating a sample incremental dataset that will be used to perform the MERGE operation.

+--------------+---------------+------+--------+
|      Order ID|       Category| Sales|Quantity|
+--------------+---------------+------+--------+
|CA-2016-152156|     Technology|1200.0|       4|
|CA-2016-138688|      Furniture| 850.0|       2|
|CA-2025-999999|Office Supplies| 300.0|       5|
+--------------+---------------+------+--------+



### 5.Merge Operation
- Before merge ,i need to make column names same in both source and destination so i have changed in the inc_df

In [0]:
from pyspark.sql.functions import col
inc_df=inc_df.select([col(c).alias(c.replace(" ", "_")) for c in inc_df.columns])
#renaming the columns as both names are different which causes error

In [0]:
dTable.alias("target") \
.merge(inc_df.alias("source"),
    "target.Order_ID=source.Order_ID"
)\
.whenMatchedUpdate(set={"Category":"source.Category",
    "Sales":"source.Sales",
    "Quantity":"source.Quantity"
})\
.whenNotMatchedInsert(values={"Order_ID":"source.Order_ID",
    "Category":"source.Category",
    "Sales":"source.Sales",
    "Quantity":"source.Quantity"
})\
.execute()
#MERGE operation to update existing records and insert new ones

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### 6.Validating the results

In [0]:
dTable.toDF().show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|     5|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|   Consumer|United States|Fort Lauderdale|       Florida|      33311|  South|OFF-ST-10000760|

In [0]:
print("Total Rows:",dTable.toDF().count())

Total Rows: 9995


### 7.Display final dataset

In [0]:
dTable.toDF().show(20,False)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------+--------+--------+--------+---------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name     |Segment    |Country      |City           |State         |Postal_Code|Region |Product_ID     |Category       |Sub-Category|Product_Name                                                       |Sales   |Quantity|Discount|Profit   |
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------+--------+--------+--------+---------+
|5     |US-2015-108966|2015-10-11|2015-10-1